# 基于 LSTM 的宋诗生成
**姓名：李伟龙**   **学号：SA25214030**

## 实验目标

本实验使用宋诗数据集训练字符级 LSTM 语言模型，实现：

1. 古诗数据读取
2. 七言绝句筛选
3. 模型训练
4. 基于起始词生成古诗

模型以“明月”为起始词生成固定格式古诗。

In [ ]:
import json
import re
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

## 数据读取

实验使用 `poet.song.40000.json` 到 `poet.song.43000.json`中的宋诗数据。

In [ ]:
data_dir = Path("data")

json_files = sorted(data_dir.glob("poet.song.*.json"))

data = []

for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        part = json.load(f)
        data.extend(part)

        print(f"{file.name}: {len(part)}")
        
print("总数据量:", len(data))
print(data[0])

all_poems = []

for item in data:
    paragraphs = item.get("paragraphs", [])
    all_poems.append(paragraphs)

print("诗歌数量:", len(all_poems))
print(all_poems[0])

## 数据预处理

为了生成固定格式古诗，本实验筛选每句七字的七言绝句。

In [ ]:
def clean_line(line):
    line = line.strip()

    line = re.sub(
        r"[，。！？、；：,.!?;:\s]",
        "",
        line
    )

    return line
BAD_CHARS = set("□{}[]（）()《》〈〉「」『』0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")

def is_qiyan_jueju(paragraphs):
    # 数据中七言绝句通常是 2 个 paragraph，每个 paragraph 包含两句七言
    if len(paragraphs) != 2:
        return False

    full_text = "".join(paragraphs)

    # 过滤缺字、夹注、英文、数字等异常字符
    if any(ch in full_text for ch in BAD_CHARS):
        return False

    for line in paragraphs:
        line = clean_line(line)

        # 每个 paragraph 去掉标点后应为 14 字
        if len(line) != 14:
            return False

    return True

qiyan_poems = []

for paragraphs in all_poems:

    if is_qiyan_jueju(paragraphs):

        poem = ""

        for line in paragraphs:
            poem += clean_line(line)

        qiyan_poems.append(poem)

print("七言绝句数量:", len(qiyan_poems))

print(qiyan_poems[:5])

START_TOKEN = "<"
END_TOKEN = ">"

poems = []

for poem in qiyan_poems:
    poems.append(
        START_TOKEN + poem + END_TOKEN
    )

print(poems[0])

## 字符词表构建

神经网络无法直接处理汉字，因此需要：字符 → 数字的映射。

In [ ]:
all_text = "".join(poems)

chars = sorted(list(set(all_text)))

char2idx = {
    char: idx
    for idx, char in enumerate(chars)
}

idx2char = {
    idx: char
    for char, idx in char2idx.items()
}

vocab_size = len(chars)

print("词表大小:", vocab_size)

print(chars[:20])

text = "明月"

ids = [char2idx[c] for c in text]

print(ids)

recover_text = "".join(
    [idx2char[i] for i in ids]
)

print(recover_text)

## Dataset 构建

训练目标：输入前面的字符，预测下一个字符。

In [ ]:
class PoetryDataset(Dataset):

    def __init__(self, poems, char2idx):

        self.samples = []

        for poem in poems:

            ids = [
                char2idx[c]
                for c in poem
            ]

            x = ids[:-1]
            y = ids[1:]

            self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        x, y = self.samples[idx]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )
    
dataset = PoetryDataset(poems,char2idx)

dataloader = DataLoader(dataset,batch_size=64,shuffle=True)

x, y = next(iter(dataloader))

print(x.shape)
print(y.shape)

## LSTM 模型

模型结构：Embedding→ LSTM→ Linear

In [ ]:
class PoetryLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        hidden_dim=256,
        num_layers=2
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x, hidden=None):

        x = self.embedding(x)

        output, hidden = self.lstm(
            x,
            hidden
        )

        logits = self.fc(output)

        return logits, hidden
    
model = PoetryLSTM(vocab_size)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

In [ ]:
num_epochs = 300

train_losses = []

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, _ = model(x)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    train_losses.append(avg_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {avg_loss:.4f}"
    )
plt.figure(figsize=(8, 5), dpi=150)

plt.plot(
    range(1, len(train_losses) + 1),
    train_losses,
    linewidth=2,
    label="Train Loss"
)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Cross Entropy Loss", fontsize=12)
plt.title("Training Loss Curve", fontsize=14)

plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
def generate_poem(
    model,
    start_text="明月",
    max_len=28,
    temperature=0.8
):

    model.eval()

    result = start_text

    input_ids = [
        char2idx[c]
        for c in START_TOKEN + start_text
    ]

    input_tensor = torch.tensor(
        [input_ids],
        dtype=torch.long
    ).to(device)

    hidden = None

    with torch.no_grad():

        logits, hidden = model(
            input_tensor,
            hidden
        )

        current_id = input_tensor[0, -1].view(1,1)

        while len(result) < max_len:

            logits, hidden = model(
                current_id,
                hidden
            )

            logits = logits[:, -1, :] / temperature

            probs = torch.softmax(
                logits,
                dim=-1
            )

            next_id = torch.multinomial(
                probs,
                num_samples=1
            )

            next_char = idx2char[
                next_id.item()
            ]

            if next_char == END_TOKEN:
                break

            result += next_char

            current_id = next_id

def format_qiyan(text):
    text = text[:28]

    lines = [
        text[:7],
        text[7:14],
        text[14:21],
        text[21:28]
    ]

    return (
        lines[0] + "，\n" +
        lines[1] + "。\n" +
        lines[2] + "，\n" +
        lines[3] + "。"
    )
for i in range(5):

    poem = generate_poem(
        model,
        start_text="明月",
        temperature=0.8
    )

    print(f"生成结果 {i+1}")
    print(format_qiyan(poem))
    print()